Load database


Lock experimental protocol: 
target class = peanut ;
train batches = 1–2 peanut only ;
validation batch = 3 almond + peanut ;
test batch = 4 réservé.


Define search space: 
matrix_method = balanced_pixels;
balanced_pixel_strategy = random / center;
preprocessings candidats ;
n_components;
règles SIMCA ;
règles empiriques CV ;
alpha;
m;
object_threshold.


Grid search classique: 
règles standard :
simple ;
alternative ;
combined_index ;
data_driven.


Grid search empirical CV: 
simple_emp_cv;
alternative_chi2_emp_cv;
alternative_empHQ_emp_cv;
data_driven_emp_cv.


Validation ranking: 
minimiser FN ;
puis minimiser FP ;
puis maximiser F1 ;
puis accuracy/balanced accuracy.


Compare best candidates: 
tableau des meilleurs modèles ;
effets du preprocessing ;
effets du nombre de composantes ;
effets de la règle ;
effets balanced_pixel_strategy.


Refit selected candidates on train: 
garder 2 ou 3 modèles candidats :
modèle haute sensibilité peanut ;
modèle compromis ;
modèle plus spécifique.


Save selected configs: 
sauvegarder une table selected_simca_configs.csv.


Sorties attendues: 
results/simca_selection/grid_standard_summary.csv
results/simca_selection/grid_empirical_cv_summary.csv
results/simca_selection/selected_simca_configs.csv
results/simca_selection/selection_errors.csv

# 04 — SIMCA model selection

This notebook selects SIMCA hyperparameters for peanut detection.

The task is formulated as a one-class detection problem:

- target class: peanut
- training data: pure peanut objects only
- projection / validation data: pure almond and pure peanut objects

The final selected configurations will be saved and evaluated later in:

- `05_simca_final_evaluation.ipynb`
- `06_mixture_projection_analysis.ipynb`

In [ ]:
from __future__ import annotations

import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 300)

# ---------------------------------------------------------------------
# Project root detection
# ---------------------------------------------------------------------
CURRENT_DIR = Path.cwd().resolve()

if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Could not find project root. Launch the notebook from the project "
        "root or from the notebooks/ folder."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

from src.io.database_h5 import load_nir_uco_h5
from src.utils import save_parquet, load_parquet, save_empty_parquet

from src.spectra.preprocessing_configs import normalize_preprocessing_configs, SIMCA_SEARCH_PREPROCESSING_CONFIGS
from src.spectra.band_selection import (
    select_wavelength_range_from_database,
    wavelength_selection_summary,
)

from src.workflows.simca_pixel_grid import (
    make_peanut_train_filters,
    run_single_simca_pixel_projection,
    run_simca_pixel_projection_grid,
    refit_best_grid_row,
)

from src.workflows.simca_cv_calibration import (
    run_simca_empirical_rule_grid,
)

from src.workflows.simca_optuna import (
    run_optuna_simca_pixel_optimization,
    best_completed_trial_row,
    refit_optuna_best_trial,
    close_optuna_study,
)

from src.decision.metrics import (
    binary_detection_metrics,
    metrics_by_group,
    summarize_pixel_errors_by_image,
)

from src.decision.aggregation import object_threshold_grid
from src.decision.uncertainty import add_three_way_object_decision

from src.visualization.plot_generic import (
    plot_bar_values,
    plot_counts_by_group,
    plot_lines_from_dataframe,
)

from src.visualization.plot_simca import (
    plot_simca_distance,
    plot_simca_rule_metric,
)

from src.visualization.plot_decision import (
    plot_pixel_error_overlay,
    plot_pixel_fp_fn_overlay,
)

%load_ext autoreload
%autoreload 2

In [ ]:
# ---------------------------------------------------------------------
# Input database
# ---------------------------------------------------------------------
DB_H5_PATH = (
    PROJECT_ROOT
    / "HSI Data"
    / "processed"
    / "nir_uco_database.h5"
)

USE_SPECTRAL_RANGE = True
SPECTRAL_MIN_NM = 1225.0
SPECTRAL_MAX_NM = 1675.0
SPECTRAL_RANGE_TAG = "1225_1675"

# ---------------------------------------------------------------------
# Optional PCA-informed preprocessing shortlist
# ---------------------------------------------------------------------
# PCA_SHORTLIST_PATH = (
#     PROJECT_ROOT
#     / "results"
#     / "pca"
#     / "pca_preprocessing_shortlist.parquet"
# )
PCA_SHORTLIST_PATH = (
    PROJECT_ROOT
    / "results"
    / f"pca_{SPECTRAL_RANGE_TAG}"
    / "pca_preprocessing_shortlist.parquet"
)

# ---------------------------------------------------------------------
# Outputs
# ---------------------------------------------------------------------
#RESULTS_DIR = PROJECT_ROOT / "results" / "simca_selection"
RESULTS_DIR = PROJECT_ROOT / "results" / f"simca_selection_{SPECTRAL_RANGE_TAG}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

WAVELENGTH_SELECTION_PATH = RESULTS_DIR / "wavelength_selection.parquet"

STANDARD_GRID_SUMMARY_PATH = RESULTS_DIR / "standard_grid_summary.parquet"
STANDARD_GRID_ERRORS_PATH = RESULTS_DIR / "standard_grid_errors.parquet"
EMPIRICAL_GRID_SUMMARY_PATH = RESULTS_DIR / "empirical_cv_grid_summary.parquet"
EMPIRICAL_GRID_ERRORS_PATH = RESULTS_DIR / "empirical_cv_grid_errors.parquet"
COMBINED_SELECTION_PATH = RESULTS_DIR / "combined_simca_selection_summary.parquet"
SELECTED_CONFIGS_PATH = RESULTS_DIR / "selected_simca_configs.parquet"
SELECTED_CONFIGS_JSON = RESULTS_DIR / "selected_simca_configs.json"
SELECTION_CONFIG_JSON = RESULTS_DIR / "simca_selection_config.json"

#STANDARD_RESULTS_PKL = RESULTS_DIR / "standard_grid_results.pkl"
#EMPIRICAL_RESULTS_PKL = RESULTS_DIR / "empirical_cv_grid_results.pkl"

OPTUNA_TRIALS_PATH = RESULTS_DIR / "optuna_trials.parquet"
#OPTUNA_STORAGE_PATH = RESULTS_DIR / "optuna_simca_selection.sqlite3"

# ---------------------------------------------------------------------
# Experimental protocol
# ---------------------------------------------------------------------
TARGET_CLASS = "peanut"

# Model-selection protocol:
# Train on pure peanut batches 1-2.
# Validate on pure almond + peanut batch 3.
# Keep batch 4 for final evaluation in notebook 05.
TRAIN_BATCHES = [1, 2]
VALIDATION_BATCHES = [3]
FINAL_TEST_BATCHES = [4]

# Do not use mixtures for selection.
USE_MIXTURES_FOR_SELECTION = False

# ---------------------------------------------------------------------
# Matrix / preprocessing / SIMCA search space
# ---------------------------------------------------------------------
MATRIX_METHODS_STANDARD = [
    "balanced_pixels",
    "object_median",
]

MATRIX_METHOD_EMPIRICAL = "balanced_pixels"

BALANCED_PIXEL_STRATEGIES = [
    "random",
    "center",
]

M_BALANCED_PIXELS = 40
REPLACE_BALANCED_PIXELS = False
RANDOM_STATE = 42

N_COMPONENTS_VALUES_STANDARD = [3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
N_COMPONENTS_VALUES_EMPIRICAL = [3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
ALPHA_VALUES_STANDARD = [0.05, 0.01]
ALPHA_EMPIRICAL = 0.05

STANDARD_RULE_NAMES = [
    "simple",
    "alternative",
    "combined_index",
    "data_driven",
]

EMPIRICAL_RULE_VARIANTS = [
    "simple_chi2",
    "simple_emp_cv",
    "alternative_chi2_fixed2",
    "alternative_chi2_emp_cv",
    "alternative_empHQ_fixed2",
    "alternative_empHQ_emp_cv",
    "data_driven_chi2",
    "data_driven_emp_cv",
    "combined_index_chi2",
]

OBJECT_THRESHOLDS = [
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
]

# Truth labels for projected pure images are direct.
POSITION_DILATION_RADIUS = 3

# Savitzky-Golay parameters
SG_WINDOW_LENGTH = 11
SG_POLYORDER = 2

# CV calibration
CV_N_SPLITS = 5
CV_GROUP_COL = "object_id"

# ---------------------------------------------------------------------
# Runtime switches
# ---------------------------------------------------------------------
RUN_STANDARD_GRID = True
RUN_EMPIRICAL_CV_GRID = True

# Optuna is optional and can be long.
RUN_OPTUNA = False
N_OPTUNA_TRIALS = 80

# Keep full pixel tables only for refitted selected models, not during full grids.
KEEP_PIXEL_TABLES_IN_GRID = False
KEEP_CV_TABLES_IN_GRID = False

print("DB_H5_PATH:", DB_H5_PATH)
print("RESULTS_DIR:", RESULTS_DIR)

In [ ]:
object_db, image_db = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=True,
)

if USE_SPECTRAL_RANGE:
    object_db, image_db, wavelengths, wavelength_info = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=SPECTRAL_MIN_NM,
        max_nm=SPECTRAL_MAX_NM,
    )

    wavelength_selection_df = wavelength_selection_summary(wavelength_info)
    save_parquet(wavelength_selection_df, WAVELENGTH_SELECTION_PATH)

    print("Spectral range selected:")
    display(wavelength_selection_df)
else:
    first_object = next(iter(object_db.values()))
    wavelengths = first_object.get("wavelengths")
    wavelengths = np.asarray(wavelengths) if wavelengths is not None else None

object_rows = []

for object_id, obj in object_db.items():
    object_rows.append({
        "object_id": object_id,
        "source_clean_key": obj.get("source_clean_key"),
        "source_image": obj.get("source_image"),
        "sample_kind": obj.get("sample_kind"),
        "object_nut_type": obj.get("object_nut_type"),
        "image_nut_type": obj.get("image_nut_type"),
        "batch": obj.get("batch"),
        "split": obj.get("split"),
        "area_pixels": obj.get("area_pixels"),
        "n_pixels": obj.get("n_pixels"),
        "n_bands": obj.get("n_bands"),
        "is_pure": obj.get("is_pure"),
        "is_mixture": obj.get("is_mixture"),
        "is_position_reference": obj.get("is_position_reference"),
    })

object_meta_df = pd.DataFrame(object_rows)

display(object_meta_df.head())

display(
    object_meta_df
    .groupby(["sample_kind", "object_nut_type", "batch"], dropna=False)
    .size()
    .reset_index(name="n_objects")
    .sort_values(["sample_kind", "object_nut_type", "batch"], na_position="last")
)

first_object = next(iter(object_db.values()))
wavelengths = first_object.get("wavelengths")

if wavelengths is not None:
    wavelengths = np.asarray(wavelengths)
    if wavelengths.size == 0:
        wavelengths = None

if wavelengths is None:
    print("No wavelength axis found. Band indices will be used.")
else:
    print("Wavelength axis found.")
    print("n_wavelengths:", len(wavelengths))
    print("first:", wavelengths[:5])
    print("last:", wavelengths[-5:])

## 1. Lock the model-selection protocol

For model selection we use:

- training set: pure peanut objects, batches 1–2;
- validation set: pure almond and pure peanut objects, batch 3;
- final test set: batch 4, not used here;
- mixtures: not used for hyperparameter selection.

In [ ]:
train_filters = make_peanut_train_filters(
    train_batches=TRAIN_BATCHES,
    split=None,
)

validation_filters = {
    "sample_kind": ["pure"],
    "object_nut_type": ["almond", "peanut"],
    "batch": VALIDATION_BATCHES,
}

final_test_filters = {
    "sample_kind": ["pure"],
    "object_nut_type": ["almond", "peanut"],
    "batch": FINAL_TEST_BATCHES,
}

mixture_filters = {
    "sample_kind": ["mixture"],
}

print("train_filters:")
print(train_filters)

print("\nvalidation_filters:")
print(validation_filters)

print("\nfinal_test_filters reserved for notebook 05:")
print(final_test_filters)

print("\nmixture_filters reserved for notebook 06:")
print(mixture_filters)

In [ ]:
def count_objects_matching_filters(meta_df: pd.DataFrame, filters: dict) -> pd.DataFrame:
    df = meta_df.copy()

    mask = pd.Series(True, index=df.index)

    for col, allowed in filters.items():
        if col not in df.columns:
            raise KeyError(f"Column not found in object metadata: {col}")

        if allowed is None:
            continue

        allowed_values = list(allowed)
        mask = mask & df[col].isin(allowed_values)

    return df[mask].copy()


train_meta_df = count_objects_matching_filters(object_meta_df, train_filters)
validation_meta_df = count_objects_matching_filters(object_meta_df, validation_filters)
final_test_meta_df = count_objects_matching_filters(object_meta_df, final_test_filters)
mixture_meta_df = count_objects_matching_filters(object_meta_df, mixture_filters)

print("Train objects:", len(train_meta_df))
display(
    train_meta_df
    .groupby(["sample_kind", "object_nut_type", "batch"], dropna=False)
    .size()
    .reset_index(name="n_objects")
)

print("Validation objects:", len(validation_meta_df))
display(
    validation_meta_df
    .groupby(["sample_kind", "object_nut_type", "batch"], dropna=False)
    .size()
    .reset_index(name="n_objects")
)

print("Final test objects reserved:", len(final_test_meta_df))
display(
    final_test_meta_df
    .groupby(["sample_kind", "object_nut_type", "batch"], dropna=False)
    .size()
    .reset_index(name="n_objects")
)

print("Mixture objects reserved:", len(mixture_meta_df))
display(
    mixture_meta_df
    .groupby(["sample_kind", "object_nut_type"], dropna=False)
    .size()
    .reset_index(name="n_objects")
)

if len(train_meta_df) == 0:
    raise RuntimeError("No training object found. Check train_filters.")

if len(validation_meta_df) == 0:
    raise RuntimeError("No validation object found. Check validation_filters.")

protocol_counts_df = pd.concat(
    [
        train_meta_df.assign(protocol_set="train"),
        validation_meta_df.assign(protocol_set="validation"),
        final_test_meta_df.assign(protocol_set="reserved_final_test"),
        mixture_meta_df.assign(protocol_set="reserved_mixtures"),
    ],
    ignore_index=True,
)

plot_counts_by_group(
    protocol_counts_df,
    group_col="protocol_set",
    category_col="object_nut_type",
    title="Object counts by protocol set and object label",
    show=True,
)

## 2. Preprocessing candidates

We start from the PCA shortlist if available.

If no PCA shortlist is available, we use a compact default list focused on meaningful NIR preprocessing chains.

In [ ]:
DEFAULT_SIMCA_PREPROCESSING_CONFIGS  = SIMCA_SEARCH_PREPROCESSING_CONFIGS

if PCA_SHORTLIST_PATH.exists():
    pca_shortlist_df = pd.read_parquet(PCA_SHORTLIST_PATH)

    display(pca_shortlist_df.head())

    if "preprocessing" in pca_shortlist_df.columns:
        shortlist_names = (
            pca_shortlist_df["preprocessing"]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .tolist()
        )
    else:
        shortlist_names = []

    preprocessing_configs = {
        name: DEFAULT_SIMCA_PREPROCESSING_CONFIGS[name]
        for name in shortlist_names
        if name in DEFAULT_SIMCA_PREPROCESSING_CONFIGS
    }

    if len(preprocessing_configs) == 0:
        print("[WARNING] PCA shortlist did not match default SIMCA configs. Using defaults.")
        preprocessing_configs = DEFAULT_SIMCA_PREPROCESSING_CONFIGS.copy()

else:
    preprocessing_configs = DEFAULT_SIMCA_PREPROCESSING_CONFIGS.copy()

preprocessing_configs = normalize_preprocessing_configs(preprocessing_configs)

preprocessing_configs_df = pd.DataFrame([
    {
        "preprocessing": name,
        "steps": " + ".join(steps),
        "n_steps": len(steps),
    }
    for name, steps in preprocessing_configs.items()
])

display(preprocessing_configs_df)
print(f"Number of preprocessing configs: {len(preprocessing_configs)}")

In [ ]:
def add_simca_selection_score(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add a scalar score for model selection.

    The hierarchy is:
    1. minimize false negatives,
    2. minimize false positives,
    3. maximize accuracy / F1 / balanced accuracy.

    Higher score is better.
    """
    out = df.copy()

    sens = out.get("peanut_sensitivity", np.nan)
    spec = out.get("almond_specificity", np.nan)

    if "fn_rate" not in out.columns:
        out["fn_rate"] = 1.0 - pd.Series(sens, index=out.index).astype(float)

    if "fp_rate" not in out.columns:
        out["fp_rate"] = 1.0 - pd.Series(spec, index=out.index).astype(float)

    f1 = out["f1_score"] if "f1_score" in out.columns else 0.0
    acc = out["accuracy"] if "accuracy" in out.columns else 0.0
    ba = out["balanced_accuracy"] if "balanced_accuracy" in out.columns else 0.0

    out["selection_score"] = (
        -10.0 * out["fn_rate"].astype(float)
        -1.0 * out["fp_rate"].astype(float)
        +0.05 * pd.Series(acc, index=out.index).astype(float).fillna(0.0)
        +0.02 * pd.Series(f1, index=out.index).astype(float).fillna(0.0)
        +0.02 * pd.Series(ba, index=out.index).astype(float).fillna(0.0)
    )

    out["selection_rank_tuple"] = list(zip(
        out["fn_rate"].round(6),
        out["fp_rate"].round(6),
        -out["accuracy"].fillna(0).round(6),
    ))

    return out


def sort_simca_selection(df: pd.DataFrame) -> pd.DataFrame:
    """Sort according to the project hierarchy."""
    if df.empty:
        return df.copy()

    df = add_simca_selection_score(df)

    sort_cols = [
        "fn_rate",
        "fp_rate",
        "accuracy",
        "f1_score",
        "balanced_accuracy",
        "selection_score",
    ]

    sort_cols = [col for col in sort_cols if col in df.columns]

    ascending = []
    for col in sort_cols:
        if col in ["fn_rate", "fp_rate"]:
            ascending.append(True)
        else:
            ascending.append(False)

    return df.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)

## 3. Standard SIMCA rule grid

This section compares standard SIMCA rules:

- `simple`
- `alternative`
- `combined_index`
- `data_driven`

The grid varies:

- preprocessing,
- number of PCA/SIMCA components,
- alpha,
- object threshold,
- balanced pixel strategy.

In [ ]:
standard_summary_parts = []
standard_results_by_strategy = {}
standard_errors_parts = []

if RUN_STANDARD_GRID:
    for balanced_pixel_strategy in BALANCED_PIXEL_STRATEGIES:
        print("=" * 100)
        print("STANDARD GRID")
        print("balanced_pixel_strategy:", balanced_pixel_strategy)
        print("=" * 100)

        standard_summary_df_i, standard_results_i, standard_errors_df_i = run_simca_pixel_projection_grid(
            object_db=object_db,
            image_db=image_db,
            matrix_methods=MATRIX_METHODS_STANDARD,
            preprocessing_configs=preprocessing_configs,
            rule_names=STANDARD_RULE_NAMES,
            train_filters=train_filters,
            projection_filters=validation_filters,
            object_thresholds=OBJECT_THRESHOLDS,
            n_components_values=N_COMPONENTS_VALUES_STANDARD,
            alpha_values=ALPHA_VALUES_STANDARD,
            m=M_BALANCED_PIXELS,
            random_state=RANDOM_STATE,
            replace=REPLACE_BALANCED_PIXELS,
            wavelengths=wavelengths,
            sg_window_length=SG_WINDOW_LENGTH,
            sg_polyorder=SG_POLYORDER,
            position_dilation_radius=POSITION_DILATION_RADIUS,
            keep_pixel_tables=KEEP_PIXEL_TABLES_IN_GRID,
            verbose=True,
            balanced_pixel_strategy=balanced_pixel_strategy,
        )

        if len(standard_summary_df_i) > 0:
            standard_summary_df_i["balanced_pixel_strategy"] = balanced_pixel_strategy
            standard_summary_df_i["search_method"] = "standard_grid"
            standard_summary_parts.append(standard_summary_df_i)

        if len(standard_errors_df_i) > 0:
            standard_errors_df_i["balanced_pixel_strategy"] = balanced_pixel_strategy
            standard_errors_parts.append(standard_errors_df_i)

        standard_results_by_strategy[balanced_pixel_strategy] = standard_results_i

    standard_summary_df = (
        pd.concat(standard_summary_parts, ignore_index=True)
        if standard_summary_parts
        else pd.DataFrame()
    )

    standard_errors_df = (
        pd.concat(standard_errors_parts, ignore_index=True)
        if standard_errors_parts
        else pd.DataFrame()
    )

else:
    standard_summary_df = pd.DataFrame()
    standard_errors_df = pd.DataFrame()
    standard_results_by_strategy = {}

standard_summary_df = sort_simca_selection(standard_summary_df)

display(standard_summary_df.head(30))
display(standard_errors_df)

In [ ]:
save_parquet(standard_summary_df, STANDARD_GRID_SUMMARY_PATH)
save_parquet(standard_errors_df, STANDARD_GRID_ERRORS_PATH)

standard_results_by_strategy = {}

print("Saved standard grid outputs:")
print(" -", STANDARD_GRID_SUMMARY_PATH)
print(" -", STANDARD_GRID_ERRORS_PATH)
print("Full standard grid objects were not saved.")

In [ ]:
standard_summary_df = load_parquet(STANDARD_GRID_SUMMARY_PATH)

In [ ]:
if len(standard_summary_df) > 0:
    plot_bar_values(
        x=(
            standard_summary_df.head(20)["preprocessing"].astype(str)
            + " | "
            + standard_summary_df.head(20)["rule"].astype(str)
            + " | A="
            + standard_summary_df.head(20)["n_components"].astype(str)
            + " | thr="
            + standard_summary_df.head(20)["object_threshold"].astype(str)
            + " | "
            + standard_summary_df.head(20)["balanced_pixel_strategy"].astype(str)
        ),
        y=standard_summary_df.head(20)["selection_score"],
        title="Top 20 standard SIMCA configurations by selection score",
        x_title="configuration",
        y_title="selection score",
        show=True,
    )

    plot_counts_by_group(
        standard_summary_df.head(100),
        group_col="balanced_pixel_strategy",
        category_col="rule",
        title="Top 100 standard configurations by strategy and rule",
        show=True,
    )
else:
    print("No standard grid result to plot.")

### Closer study

#### by best balanced accuracy

In [ ]:
standard_summary_df[standard_summary_df['balanced_accuracy']>=0.85].sort_values('balanced_accuracy', ascending=False)

In [ ]:
standard_best_ba = standard_summary_df.filter(items=[1049, 1541, 1392, 1145, 1889], axis=0)
standard_best_ba

#### By best FN rate

In [ ]:
standard_summary_df[(standard_summary_df['fn']==0)&(standard_summary_df['non_target_specificity']>=0.5)].sort_values('fp_rate')

In [ ]:
standard_best_fn = standard_summary_df.filter(items=[0, 1, 2, 3, 4], axis=0)
standard_best_fn

In [ ]:
standard_best = pd.concat([standard_best_ba, standard_best_fn]).drop_duplicates().sort_values('selection_score', ascending=False)
standard_best

## 4. Empirical CV SIMCA rule grid

This section uses grouped cross-validation on pure peanut training objects to calibrate empirical thresholds.

It then evaluates several rule variants on the validation set.

This is especially useful because theoretical SIMCA thresholds can be too strict or too permissive depending on preprocessing, noise level and the number of training objects.

In [ ]:
RUN_EMPIRICAL_CV_GRID = True

In [ ]:
empirical_summary_parts = []
empirical_results_by_strategy = {}
empirical_errors_parts = []

if RUN_EMPIRICAL_CV_GRID:
    for balanced_pixel_strategy in BALANCED_PIXEL_STRATEGIES:
        print("=" * 100)
        print("EMPIRICAL CV RULE GRID")
        print("balanced_pixel_strategy:", balanced_pixel_strategy)
        print("=" * 100)

        empirical_summary_df_i, empirical_results_i, empirical_errors_df_i = run_simca_empirical_rule_grid(
            object_db=object_db,
            image_db=image_db,
            train_filters=train_filters,
            projection_filters=validation_filters,
            preprocessing_configs=preprocessing_configs,
            rule_variants=EMPIRICAL_RULE_VARIANTS,
            n_components_values=N_COMPONENTS_VALUES_EMPIRICAL,
            matrix_method=MATRIX_METHOD_EMPIRICAL,
            #alpha=ALPHA_EMPIRICAL,
            alpha=0.01,
            object_threshold=0.75,
            m=M_BALANCED_PIXELS,
            random_state=RANDOM_STATE,
            replace=REPLACE_BALANCED_PIXELS,
            wavelengths=wavelengths,
            sg_window_length=SG_WINDOW_LENGTH,
            sg_polyorder=SG_POLYORDER,
            position_dilation_radius=POSITION_DILATION_RADIUS,
            cv_n_splits=CV_N_SPLITS,
            group_col=CV_GROUP_COL,
            keep_pixel_tables=KEEP_PIXEL_TABLES_IN_GRID,
            keep_cv_tables=KEEP_CV_TABLES_IN_GRID,
            verbose=True,
            balanced_pixel_strategy=balanced_pixel_strategy,
        )

        if len(empirical_summary_df_i) > 0:
            empirical_summary_df_i["balanced_pixel_strategy"] = balanced_pixel_strategy
            empirical_summary_df_i["search_method"] = "empirical_cv_grid"
            empirical_summary_parts.append(empirical_summary_df_i)

        if len(empirical_errors_df_i) > 0:
            empirical_errors_df_i["balanced_pixel_strategy"] = balanced_pixel_strategy
            empirical_errors_parts.append(empirical_errors_df_i)

        empirical_results_by_strategy[balanced_pixel_strategy] = empirical_results_i

    empirical_summary_df = (
        pd.concat(empirical_summary_parts, ignore_index=True)
        if empirical_summary_parts
        else pd.DataFrame()
    )

    empirical_errors_df = (
        pd.concat(empirical_errors_parts, ignore_index=True)
        if empirical_errors_parts
        else pd.DataFrame()
    )

else:
    empirical_summary_df = pd.DataFrame()
    empirical_errors_df = pd.DataFrame()
    empirical_results_by_strategy = {}

empirical_summary_df = sort_simca_selection(empirical_summary_df)

display(empirical_summary_df.head(30))
display(empirical_errors_df)

In [ ]:
empirical_summary_1_df = empirical_summary_df.copy()

In [ ]:
save_parquet(empirical_summary_df, EMPIRICAL_GRID_SUMMARY_PATH)
save_parquet(empirical_errors_df, EMPIRICAL_GRID_ERRORS_PATH)

# Do not save full grid objects.
empirical_results_by_strategy = {}

print("Saved empirical CV grid outputs:")
print(" -", EMPIRICAL_GRID_SUMMARY_PATH)
print(" -", EMPIRICAL_GRID_ERRORS_PATH)
print("Full empirical grid objects were not saved.")

In [ ]:
empirical_summary_df = load_parquet(EMPIRICAL_GRID_SUMMARY_PATH)
empirical_summary_df

In [ ]:
if len(empirical_summary_df) > 0:
    plot_bar_values(
        x=(
            empirical_summary_df.head(20)["preprocessing"].astype(str)
            + " | "
            + empirical_summary_df.head(20)["rule_variant"].astype(str)
            + " | A="
            + empirical_summary_df.head(20)["n_components"].astype(str)
            + " | "
            + empirical_summary_df.head(20)["balanced_pixel_strategy"].astype(str)
        ),
        y=empirical_summary_df.head(20)["selection_score"],
        title="Top 20 empirical CV SIMCA configurations by selection score",
        x_title="configuration",
        y_title="selection score",
        show=True,
    )

    plot_counts_by_group(
        empirical_summary_df.head(100),
        group_col="balanced_pixel_strategy",
        category_col="rule_variant",
        title="Top 100 empirical CV configurations by strategy and rule variant",
        show=True,
    )
else:
    print("No empirical CV grid result to plot.")

### Closer study

In [ ]:
empirical_summary_df.columns

In [ ]:
empirical_cols = [
    'n', 'tp', 'fn', 'fp', 'tn', 'target_sensitivity',
    'non_target_specificity', 'balanced_accuracy', 'accuracy', 'precision', 'f1_score', 'object_threshold', 'matrix_method',
    'preprocessing', 'rule_variant', 'n_components', 'alpha', 'balanced_pixel_strategy',
    'selection_score',
]

#### By best balanced accuracy

In [ ]:
empirical_summary_1_df[empirical_summary_1_df['balanced_accuracy']>=0.85][empirical_cols].sort_values('balanced_accuracy', ascending=False)

In [ ]:
empirical_best_ba_1 = empirical_summary_1_df[empirical_summary_1_df['balanced_accuracy']>=0.85].sort_values('balanced_accuracy', ascending=False)

In [ ]:
empirical_summary_df[empirical_summary_df['balanced_accuracy']>=0.85][empirical_cols].sort_values('balanced_accuracy', ascending=False)

In [ ]:
empirical_best_ba = empirical_summary_df[empirical_summary_df['balanced_accuracy']>=0.85].sort_values('balanced_accuracy', ascending=False)

#### By best FN rate

In [ ]:
empirical_summary_1_df[(empirical_summary_1_df['fn']==0)&(empirical_summary_1_df['non_target_specificity']>=0.5)][empirical_cols].sort_values('fp')

In [ ]:
empirical_best_fn_1 = empirical_summary_1_df[(empirical_summary_1_df['fn']==0)&(empirical_summary_1_df['non_target_specificity']>=0.5)].sort_values('fp').head(5)
empirical_best_fn_1

In [ ]:
empirical_summary_df[(empirical_summary_df['fn']==0)&(empirical_summary_df['non_target_specificity']>=0.5)][empirical_cols].sort_values('fp')

In [ ]:
empirical_best_fn = empirical_summary_df[(empirical_summary_df['fn']==0)&(empirical_summary_df['non_target_specificity']>=0.5)].sort_values('fp').head(5)
empirical_best_fn

In [ ]:
empirical_best = pd.concat([empirical_best_ba, empirical_best_fn, empirical_best_ba_1, empirical_best_fn_1]).drop_duplicates().sort_values('selection_score', ascending=False).reset_index(drop=True)
empirical_best

## 5. Optional Optuna search

This section is optional.

It can be used to compare the manual grid search with a Bayesian optimization strategy. By default, `RUN_OPTUNA = False` because this can take time.

In [ ]:
RUN_OPTUNA = True

In [ ]:
if RUN_OPTUNA:
    study, optuna_trials_df = run_optuna_simca_pixel_optimization(
        object_db=object_db,
        image_db=image_db,
        train_filters=train_filters,
        projection_filters=validation_filters,
        matrix_methods=["balanced_pixels"],
        preprocessing_configs=preprocessing_configs,
        rule_names=STANDARD_RULE_NAMES,
        n_components_choices=N_COMPONENTS_VALUES_STANDARD,
        alpha_choices=ALPHA_VALUES_STANDARD,
        n_trials=N_OPTUNA_TRIALS,
        timeout=None,
        study_name="simca_model_selection_batch3_validation",
        storage_path=None,
        load_if_exists=True,
        random_state=RANDOM_STATE,
        n_jobs=1,
        show_progress_bar=True,
        close_storage=True,
        balanced_pixel_strategy_choices=BALANCED_PIXEL_STRATEGIES,
        object_threshold_low=min(OBJECT_THRESHOLDS),
        object_threshold_high=max(OBJECT_THRESHOLDS),
        object_threshold_step=0.05,
        m_choices=[M_BALANCED_PIXELS],
        replace=REPLACE_BALANCED_PIXELS,
        wavelengths=wavelengths,
        sg_window_choices=[SG_WINDOW_LENGTH],
        sg_polyorder_choices=[SG_POLYORDER],
        position_dilation_radius_choices=[POSITION_DILATION_RADIUS],
        objective_metric="fn_fp_hierarchical",
        min_peanut_sensitivity=0.50,
        min_almond_specificity=0.10,
    )

    save_parquet(optuna_trials_df, OPTUNA_TRIALS_PATH)
    display(optuna_trials_df.head(20))

    close_optuna_study(study)

else:
    optuna_trials_df = pd.DataFrame()
    study = None
    print("Optuna search skipped.")

## 6. Combine and compare all selection results

We now combine:

- standard grid results,
- empirical CV rule grid results,
- optional Optuna results if available.

The ranking follows the project priority:

1. minimize false negatives,
2. minimize false positives,
3. maximize F1 / accuracy / balanced accuracy.

In [ ]:
empirical_summary_df.rule_variant.unique()

In [ ]:
SIMCA_RULE_METADATA = {
    # ------------------------------------------------------------------
    # Standard rules
    # ------------------------------------------------------------------
    "simple": {
        "rule_base": "simple",
        "rule_variant": "simple_chi2",
        "rule_display": "simple",
        "statistic_family": "max_HQ",
        "normalization": "chi2_HQ_limits",
        "limit_source": "fixed_1",
        "uses_cv_threshold": False,
        "uses_empirical_HQ_limits": False,
        "standard_equivalent": "simple",
    },
    "alternative": {
        "rule_base": "alternative",
        "rule_variant": "alternative_chi2_fixed2",
        "rule_display": "alternative",
        "statistic_family": "sum_HQ",
        "normalization": "chi2_HQ_limits",
        "limit_source": "fixed_2",
        "uses_cv_threshold": False,
        "uses_empirical_HQ_limits": False,
        "standard_equivalent": "alternative",
    },
    "data_driven": {
        "rule_base": "data_driven",
        "rule_variant": "data_driven_chi2",
        "rule_display": "data_driven",
        "statistic_family": "data_driven_D",
        "normalization": "model_moments",
        "limit_source": "chi2",
        "uses_cv_threshold": False,
        "uses_empirical_HQ_limits": False,
        "standard_equivalent": "data_driven",
    },
    "combined_index": {
        "rule_base": "combined_index",
        "rule_variant": "combined_index_chi2",
        "rule_display": "combined_index",
        "statistic_family": "combined_index_C",
        "normalization": "chi2_HQ_limits",
        "limit_source": "scaled_chi2",
        "uses_cv_threshold": False,
        "uses_empirical_HQ_limits": False,
        "standard_equivalent": "combined_index",
    },

    # ------------------------------------------------------------------
    # Empirical CV variants
    # ------------------------------------------------------------------
    "simple_chi2": {
        "rule_base": "simple",
        "rule_variant": "simple_chi2",
        "rule_display": "simple_chi2",
        "statistic_family": "max_HQ",
        "normalization": "chi2_HQ_limits",
        "limit_source": "fixed_1",
        "uses_cv_threshold": False,
        "uses_empirical_HQ_limits": False,
        "standard_equivalent": "simple",
    },
    "simple_emp_cv": {
        "rule_base": "simple",
        "rule_variant": "simple_emp_cv",
        "rule_display": "simple_emp_cv",
        "statistic_family": "max_HQ",
        "normalization": "chi2_HQ_limits",
        "limit_source": "empirical_cv",
        "uses_cv_threshold": True,
        "uses_empirical_HQ_limits": False,
        "standard_equivalent": "simple",
    },
    "alternative_chi2_fixed2": {
        "rule_base": "alternative",
        "rule_variant": "alternative_chi2_fixed2",
        "rule_display": "alternative_chi2_fixed2",
        "statistic_family": "sum_HQ",
        "normalization": "chi2_HQ_limits",
        "limit_source": "fixed_2",
        "uses_cv_threshold": False,
        "uses_empirical_HQ_limits": False,
        "standard_equivalent": "alternative",
    },
    "alternative_chi2_emp_cv": {
        "rule_base": "alternative",
        "rule_variant": "alternative_chi2_emp_cv",
        "rule_display": "alternative_chi2_emp_cv",
        "statistic_family": "sum_HQ",
        "normalization": "chi2_HQ_limits",
        "limit_source": "empirical_cv",
        "uses_cv_threshold": True,
        "uses_empirical_HQ_limits": False,
        "standard_equivalent": "alternative",
    },
    "alternative_empHQ_fixed2": {
        "rule_base": "alternative",
        "rule_variant": "alternative_empHQ_fixed2",
        "rule_display": "alternative_empHQ_fixed2",
        "statistic_family": "sum_HQ",
        "normalization": "empirical_HQ_limits",
        "limit_source": "fixed_2",
        "uses_cv_threshold": True,
        "uses_empirical_HQ_limits": True,
        "standard_equivalent": "alternative",
    },
    "alternative_empHQ_emp_cv": {
        "rule_base": "alternative",
        "rule_variant": "alternative_empHQ_emp_cv",
        "rule_display": "alternative_empHQ_emp_cv",
        "statistic_family": "sum_HQ",
        "normalization": "empirical_HQ_limits",
        "limit_source": "empirical_cv",
        "uses_cv_threshold": True,
        "uses_empirical_HQ_limits": True,
        "standard_equivalent": "alternative",
    },
    "data_driven_chi2": {
        "rule_base": "data_driven",
        "rule_variant": "data_driven_chi2",
        "rule_display": "data_driven_chi2",
        "statistic_family": "data_driven_D",
        "normalization": "model_moments",
        "limit_source": "chi2",
        "uses_cv_threshold": False,
        "uses_empirical_HQ_limits": False,
        "standard_equivalent": "data_driven",
    },
    "data_driven_emp_cv": {
        "rule_base": "data_driven",
        "rule_variant": "data_driven_emp_cv",
        "rule_display": "data_driven_emp_cv",
        "statistic_family": "data_driven_D",
        "normalization": "model_moments",
        "limit_source": "empirical_cv",
        "uses_cv_threshold": True,
        "uses_empirical_HQ_limits": False,
        "standard_equivalent": "data_driven",
    },
    "combined_index_chi2": {
        "rule_base": "combined_index",
        "rule_variant": "combined_index_chi2",
        "rule_display": "combined_index_chi2",
        "statistic_family": "combined_index_C",
        "normalization": "chi2_HQ_limits",
        "limit_source": "scaled_chi2",
        "uses_cv_threshold": False,
        "uses_empirical_HQ_limits": False,
        "standard_equivalent": "combined_index",
    },
}


def _is_missing_value(x) -> bool:
    try:
        return pd.isna(x)
    except Exception:
        return False


def _first_available_value(row: pd.Series, columns: list[str], default=None):
    for col in columns:
        if col in row.index:
            value = row[col]
            if not _is_missing_value(value):
                return value
    return default


def infer_model_family_from_rule_token(rule_token: str) -> str:
    """
    Infer broad model family when it is not already available.
    """
    rule_token = str(rule_token)

    if rule_token in STANDARD_RULE_NAMES:
        return "standard_rule"

    if rule_token in EMPIRICAL_RULE_VARIANTS:
        return "empirical_cv_rule"

    return "unknown"


def normalize_simca_rule_columns(
    df: pd.DataFrame,
    model_family: str | None = None,
) -> pd.DataFrame:
    """
    Normalize rule columns between standard SIMCA grid and empirical-CV grid.

    Output convention
    -----------------
    rule_original:
        Original value of column rule.

    rule_variant_original:
        Original value of column rule_variant.

    rule:
        Base SIMCA rule used for high-level comparison:
        simple, alternative, data_driven, combined_index.

    rule_variant:
        Exact implemented variant:
        simple_chi2, simple_emp_cv, alternative_chi2_fixed2, etc.

    rule_for_refit:
        Column to use later:
        - standard_rule: use rule
        - empirical_cv_rule: use rule_variant
    """
    if df is None or len(df) == 0:
        return pd.DataFrame() if df is None else df.copy()

    out = df.copy()

    if "rule" in out.columns:
        out["rule_original"] = out["rule"].astype("object")
    else:
        out["rule_original"] = np.nan

    if "rule_variant" in out.columns:
        out["rule_variant_original"] = out["rule_variant"].astype("object")
    else:
        out["rule_variant_original"] = np.nan

    normalized_rows = []

    for _, row in out.iterrows():
        # Priority:
        # empirical CV rows should use rule_variant when available;
        # standard rows generally only have rule.
        raw_token = _first_available_value(
            row,
            ["rule_variant", "rule", "selected_rule_name"],
            default=None,
        )

        if raw_token is None:
            raise ValueError("Could not infer SIMCA rule token from row.")

        raw_token = str(raw_token)

        if raw_token not in SIMCA_RULE_METADATA:
            raise ValueError(
                f"Unknown SIMCA rule token: {raw_token!r}. "
                "Add it to SIMCA_RULE_METADATA."
            )

        meta = dict(SIMCA_RULE_METADATA[raw_token])

        if model_family is not None:
            family = model_family
        elif "model_family" in row.index and not _is_missing_value(row["model_family"]):
            family = str(row["model_family"])
        else:
            family = infer_model_family_from_rule_token(raw_token)

        meta["model_family"] = family
        meta["rule_token"] = raw_token

        # Useful for exact comparisons between standard grid and empirical variants.
        meta["is_standard_equivalent_variant"] = (
            meta["rule_variant"]
            in {
                "simple_chi2",
                "alternative_chi2_fixed2",
                "data_driven_chi2",
                "combined_index_chi2",
            }
        )

        # This is the column to use for refitting later.
        if family == "standard_rule":
            meta["rule_for_refit"] = meta["rule_base"]
        elif family == "empirical_cv_rule":
            meta["rule_for_refit"] = meta["rule_variant"]
        else:
            meta["rule_for_refit"] = raw_token

        normalized_rows.append(meta)

    meta_df = pd.DataFrame(normalized_rows, index=out.index)

    # Attach metadata.
    for col in meta_df.columns:
        out[col] = meta_df[col]

    # Important convention:
    # `rule` becomes the comparable high-level rule.
    # `rule_variant` becomes the exact variant.
    out["rule"] = out["rule_base"]
    out["rule_variant"] = out["rule_variant"]

    # Keep selected_rule_name readable.
    # For standard rows: simple / alternative / ...
    # For empirical rows: exact variant.
    out["selected_rule_name"] = np.where(
        out["model_family"].eq("standard_rule"),
        out["rule"],
        out["rule_variant"],
    )

    return out

In [ ]:
standard_summary_df = normalize_simca_rule_columns(
    standard_summary_df,
    model_family="standard_rule",
)

empirical_summary_df = normalize_simca_rule_columns(
    empirical_summary_df,
    model_family="empirical_cv_rule",
)

combined_summary_df = pd.concat(
    [
        standard_summary_df,
        empirical_summary_df,
    ],
    ignore_index=True,
    sort=False,
)

combined_summary_df = normalize_simca_rule_columns(combined_summary_df)

display(
    combined_summary_df[
        [
            "model_family",
            "rule_original",
            "rule_variant_original",
            "rule",
            "rule_variant",
            "selected_rule_name",
            "rule_for_refit",
            "statistic_family",
            "normalization",
            "limit_source",
            "uses_cv_threshold",
            "uses_empirical_HQ_limits",
            "preprocessing",
            "n_components",
            "object_threshold",
            "balanced_accuracy",
            "peanut_sensitivity",
            "almond_specificity",
            "fn_rate",
            "fp_rate",
        ]
    ].head(20)
)

In [ ]:
standard_best_df = normalize_simca_rule_columns(
    standard_best,
    model_family="standard_rule",
)

empirical_best_df = normalize_simca_rule_columns(
    empirical_best,
    model_family="empirical_cv_rule",
)

combined_best = pd.concat(
    [
        standard_best_df,
        empirical_best_df,
    ],
    ignore_index=True,
    sort=False,
)

combined_best = normalize_simca_rule_columns(combined_best)

combined_best[
        [
            "model_family",
            "rule_original",
            "rule_variant_original",
            "rule",
            "rule_variant",
            "selected_rule_name",
            "rule_for_refit",
            "statistic_family",
            "normalization",
            "limit_source",
            "uses_cv_threshold",
            "uses_empirical_HQ_limits",
            "preprocessing",
            "n_components",
            "object_threshold",
            "balanced_accuracy",
            "peanut_sensitivity",
            "almond_specificity",
            "fn_rate",
            "fp_rate",
        ]
]

In [ ]:
def existing_cols(df, cols):
    return [c for c in cols if c in df.columns]

df = combined_best.copy()
df = df.reset_index(drop=False).rename(columns={"index": "original_row"})

EQUIVALENT_RULE_KEY_COLS = existing_cols(
    df,
    [
        "matrix_method",
        "balanced_pixel_strategy",
        "preprocessing",
        "preprocessing_steps",
        "rule_variant",
        "statistic_family",
        "normalization",
        "limit_source",
        "uses_cv_threshold",
        "uses_empirical_HQ_limits",
        "n_components",
        "alpha",
        "m",
        "object_threshold",
        "sg_window_length",
        "sg_polyorder",
        "position_dilation_radius",
    ],
)
metric_cols = existing_cols(
    df,
    [
        "balanced_accuracy",
        "peanut_sensitivity",
        "almond_specificity",
        "fn_rate",
        "fp_rate",
        "f1_score",
        "accuracy",
    ],
)

duplicate_group_summary_df = (
    df
    .groupby(EQUIVALENT_RULE_KEY_COLS, dropna=False)
    .agg(
        n_rows=("original_row", "count"),
        rows=("original_row", lambda x: list(x)),
        model_families=("model_family", lambda x: sorted(set(map(str, x)))),
        selected_rule_names=("selected_rule_name", lambda x: sorted(set(map(str, x)))),
        rule_for_refit_values=("rule_for_refit", lambda x: sorted(set(map(str, x)))),
        **{
            f"{col}_min": (col, "min")
            for col in metric_cols
        },
        **{
            f"{col}_max": (col, "max")
            for col in metric_cols
        },
    )
    .reset_index()
)

duplicate_group_summary_df = duplicate_group_summary_df[
    duplicate_group_summary_df["n_rows"] > 1
].copy()

display(
    duplicate_group_summary_df
    .sort_values(["n_rows", "balanced_accuracy_max"], ascending=[False, False])
)

In [ ]:
COMBINED_BEST_PATH = RESULTS_DIR / "selected_simca_candidates.parquet"
save_parquet(combined_best, COMBINED_BEST_PATH)

In [ ]:
summary_parts = []

if len(standard_summary_df) > 0:
    standard_for_combined = standard_summary_df.copy()

    if "rule_variant" not in standard_for_combined.columns:
        standard_for_combined["rule_variant"] = np.nan

    standard_for_combined["model_family"] = "standard_rule"
    standard_for_combined["selected_rule_name"] = standard_for_combined["rule"].astype(str)

    summary_parts.append(standard_for_combined)

if len(empirical_summary_df) > 0:
    empirical_for_combined = empirical_summary_df.copy()

    if "rule" not in empirical_for_combined.columns:
        empirical_for_combined["rule"] = empirical_for_combined["rule_variant"].astype(str)

    empirical_for_combined["model_family"] = "empirical_cv_rule"
    empirical_for_combined["selected_rule_name"] = empirical_for_combined["rule_variant"].astype(str)

    summary_parts.append(empirical_for_combined)

combined_summary_df = (
    pd.concat(summary_parts, ignore_index=True, sort=False)
    if summary_parts
    else pd.DataFrame()
)

combined_summary_df = sort_simca_selection(combined_summary_df)

display(combined_summary_df.head(30))

save_parquet(combined_summary_df, COMBINED_SELECTION_PATH)
print("Saved combined selection summary:")
print(COMBINED_SELECTION_PATH)

In [ ]:
ranking_cols = [
    "model_family",
    "search_method",
    "matrix_method",
    "balanced_pixel_strategy",
    "preprocessing",
    "preprocessing_steps",
    "selected_rule_name",
    "rule",
    "rule_variant",
    "n_components",
    "alpha",
    "m",
    "object_threshold",
    "sg_window_length",
    "sg_polyorder",
    "position_dilation_radius",
    "n",
    "tp",
    "fn",
    "fp",
    "tn",
    "peanut_sensitivity",
    "almond_specificity",
    "balanced_accuracy",
    "accuracy",
    "precision",
    "f1_score",
    "fn_rate",
    "fp_rate",
    "selection_score",
    "min_batch_peanut_sensitivity",
    "min_batch_almond_specificity",
    "batch3_peanut_sensitivity",
    "batch3_almond_specificity",
    "cv_target_rejection_rate",
    "cv_abs_rejection_error",
    "H_emp_over_chi2",
    "Q_emp_over_chi2",
    "D_emp_over_chi2",
]

available_ranking_cols = [
    col for col in ranking_cols
    if col in combined_summary_df.columns
]

display(combined_summary_df[available_ranking_cols].head(40))

In [ ]:
if len(combined_summary_df) > 0:
    plot_counts_by_group(
        combined_summary_df.head(100),
        group_col="model_family",
        category_col="selected_rule_name",
        title="Top 100 selected configurations by model family and rule",
        show=True,
    )

    plot_counts_by_group(
        combined_summary_df.head(100),
        group_col="balanced_pixel_strategy",
        category_col="preprocessing",
        title="Top 100 selected configurations by balanced pixel strategy and preprocessing",
        show=True,
    )
else:
    print("No combined result to plot.")

## 7. Select candidate configurations

We select a small number of configurations for final evaluation.

Recommended candidates:

1. Best global configuration according to the hierarchical score.
2. Best empirical CV rule configuration.
3. Best standard rule configuration.
4. Best high-sensitivity configuration with zero or minimal false negatives.
5. Best balanced compromise.

In [ ]:
selected_rows = []

if len(combined_summary_df) == 0:
    raise RuntimeError("No model selection result available.")

# 1. Best global
best_global = combined_summary_df.iloc[0].copy()
best_global["selection_profile"] = "best_global"
selected_rows.append(best_global)

# 2. Best empirical CV
empirical_candidates = combined_summary_df[
    combined_summary_df["model_family"].eq("empirical_cv_rule")
].copy()

if len(empirical_candidates) > 0:
    best_empirical = empirical_candidates.iloc[0].copy()
    best_empirical["selection_profile"] = "best_empirical_cv"
    selected_rows.append(best_empirical)

# 3. Best standard
standard_candidates = combined_summary_df[
    combined_summary_df["model_family"].eq("standard_rule")
].copy()

if len(standard_candidates) > 0:
    best_standard = standard_candidates.iloc[0].copy()
    best_standard["selection_profile"] = "best_standard_rule"
    selected_rows.append(best_standard)

# 4. High sensitivity: minimal FN, then minimal FP
high_sensitivity_candidates = combined_summary_df.copy()
high_sensitivity_candidates = high_sensitivity_candidates.sort_values(
    ["fn", "fp", "f1_score", "accuracy", "selection_score"],
    ascending=[True, True, False, False, False],
).reset_index(drop=True)

best_high_sensitivity = high_sensitivity_candidates.iloc[0].copy()
best_high_sensitivity["selection_profile"] = "high_sensitivity_min_fn"
selected_rows.append(best_high_sensitivity)

# 5. Balanced compromise: maximize balanced accuracy among low-FN candidates
min_fn = combined_summary_df["fn"].min()
low_fn_candidates = combined_summary_df[
    combined_summary_df["fn"].eq(min_fn)
].copy()

balanced_compromise = low_fn_candidates.sort_values(
    ["balanced_accuracy", "f1_score", "accuracy", "selection_score"],
    ascending=False,
).iloc[0].copy()

balanced_compromise["selection_profile"] = "balanced_compromise"
selected_rows.append(balanced_compromise)

selected_configs_df = (
    pd.DataFrame(selected_rows)
    .drop_duplicates(
        subset=[
            "model_family",
            "matrix_method",
            "balanced_pixel_strategy",
            "preprocessing",
            "selected_rule_name",
            "n_components",
            "alpha",
            "object_threshold",
        ]
    )
    .reset_index(drop=True)
)

display(selected_configs_df[["selection_profile"] + available_ranking_cols])

In [ ]:
save_parquet(selected_configs_df, SELECTED_CONFIGS_PATH)
selected_records = selected_configs_df.replace({np.nan: None}).to_dict(orient="records")

with open(SELECTED_CONFIGS_JSON, "w", encoding="utf-8") as f:
    json.dump(selected_records, f, indent=2)

print("Saved selected configurations:")
print(" -", SELECTED_CONFIGS_PATH)
print(" -", SELECTED_CONFIGS_JSON)

In [ ]:
def fill_selected_config_defaults(selected_configs_df: pd.DataFrame) -> pd.DataFrame:
    """
    Fill missing values needed for refit.

    Standard-grid rows may not contain sg_window_length, sg_polyorder
    or position_dilation_radius because these parameters were fixed during
    the search and not saved in the summary table.
    """
    df = selected_configs_df.copy()

    default_values = {
        "sg_window_length": SG_WINDOW_LENGTH,
        "sg_polyorder": SG_POLYORDER,
        "position_dilation_radius": POSITION_DILATION_RADIUS,
        "m": M_BALANCED_PIXELS,
        "balanced_pixel_strategy": "random",
        "matrix_method": "balanced_pixels",
        "alpha": 0.05,
        "object_threshold": 0.75,
    }

    for col, default in default_values.items():
        if col not in df.columns:
            df[col] = default
        df[col] = df[col].fillna(default)

    # For standard-rule rows, rule_variant is often NaN.
    # Fill it for convenience, but it will only be used for empirical rows.
    if "rule_variant" not in df.columns:
        df["rule_variant"] = np.nan

    if "selected_rule_name" not in df.columns:
        df["selected_rule_name"] = np.nan

    df["rule_variant"] = df["rule_variant"].fillna(df["selected_rule_name"])

    # Make numeric columns explicit.
    int_cols = [
        "n_components",
        "sg_window_length",
        "sg_polyorder",
        "position_dilation_radius",
        "m",
    ]

    for col in int_cols:
        if col in df.columns:
            df[col] = df[col].astype(float).astype(int)

    float_cols = [
        "alpha",
        "object_threshold",
    ]

    for col in float_cols:
        if col in df.columns:
            df[col] = df[col].astype(float)

    return df


selected_configs_df = fill_selected_config_defaults(selected_configs_df)

display(
    selected_configs_df[
        [
            "selection_profile",
            "model_family",
            "preprocessing",
            "selected_rule_name",
            "rule",
            "rule_variant",
            "n_components",
            "alpha",
            "m",
            "object_threshold",
            "sg_window_length",
            "sg_polyorder",
            "position_dilation_radius",
            "balanced_pixel_strategy",
        ]
    ]
)

## 8. Refit selected configurations on validation for detailed diagnostics

We refit the selected configurations and keep full pixel/object tables.

This is still validation diagnostics only. The final test batch is not used here.

In [ ]:
def refit_selected_config(
    row: pd.Series,
    object_db,
    image_db,
    train_filters: dict,
    validation_filters: dict,
    preprocessing_configs: dict,
    wavelengths=None,
):
    """
    Refit a selected configuration and return a full result dictionary.

    Standard-rule rows use `refit_best_grid_row`.

    Empirical-CV-rule rows are refit through `run_simca_empirical_rule_grid`
    with a single preprocessing / n_components / rule_variant configuration,
    keeping pixel tables.
    """
    model_family = str(row["model_family"])

    if model_family == "standard_rule":
        return refit_best_grid_row(
            object_db=object_db,
            image_db=image_db,
            best_row=row,
            train_filters=train_filters,
            projection_filters=validation_filters,
            preprocessing_configs=preprocessing_configs,
            object_thresholds=[float(row["object_threshold"])],
            m=int(row.get("m", M_BALANCED_PIXELS))
                if pd.notna(row.get("m", np.nan))
                else M_BALANCED_PIXELS,
            random_state=RANDOM_STATE,
            replace=REPLACE_BALANCED_PIXELS,
            wavelengths=wavelengths,
            sg_window_length=int(row.get("sg_window_length", SG_WINDOW_LENGTH)),
            sg_polyorder=int(row.get("sg_polyorder", SG_POLYORDER)),
            position_dilation_radius=int(row.get("position_dilation_radius", POSITION_DILATION_RADIUS)),
            balanced_pixel_strategy=str(row.get("balanced_pixel_strategy", "random")),
        )

    if model_family == "empirical_cv_rule":
        preprocessing_name = str(row["preprocessing"])

        single_preproc = {
            preprocessing_name: preprocessing_configs[preprocessing_name]
        }

        single_summary, single_results, single_errors = run_simca_empirical_rule_grid(
            object_db=object_db,
            image_db=image_db,
            train_filters=train_filters,
            projection_filters=validation_filters,
            preprocessing_configs=single_preproc,
            rule_variants=[str(row["rule_variant"])],
            n_components_values=[int(row["n_components"])],
            matrix_method=str(row["matrix_method"]),
            alpha=float(row["alpha"]),
            object_threshold=float(row["object_threshold"]),
            m=int(row.get("m", M_BALANCED_PIXELS))
                if pd.notna(row.get("m", np.nan))
                else M_BALANCED_PIXELS,
            random_state=RANDOM_STATE,
            replace=REPLACE_BALANCED_PIXELS,
            wavelengths=wavelengths,
            sg_window_length=int(row.get("sg_window_length", SG_WINDOW_LENGTH)),
            sg_polyorder=int(row.get("sg_polyorder", SG_POLYORDER)),
            position_dilation_radius=int(row.get("position_dilation_radius", POSITION_DILATION_RADIUS)),
            cv_n_splits=CV_N_SPLITS,
            group_col=CV_GROUP_COL,
            keep_pixel_tables=True,
            keep_cv_tables=True,
            verbose=True,
            balanced_pixel_strategy=str(row.get("balanced_pixel_strategy", "random")),
        )

        return {
            "summary_df": single_summary,
            "results": single_results,
            "errors_df": single_errors,
        }

    raise ValueError(f"Unknown model_family: {model_family}")

In [ ]:
selected_refit_results = {}

for idx, row in selected_configs_df.iterrows():
    profile = str(row["selection_profile"])

    print("=" * 100)
    print("Refitting selected profile:", profile)
    print("model_family:", row["model_family"])
    print("preprocessing:", row["preprocessing"])
    print("rule:", row["selected_rule_name"])
    print("=" * 100)

    selected_refit_results[profile] = refit_selected_config(
        row=row,
        object_db=object_db,
        image_db=image_db,
        train_filters=train_filters,
        validation_filters=validation_filters,
        preprocessing_configs=preprocessing_configs,
        wavelengths=wavelengths,
    )

In [ ]:
validation_object_tables = {}

for profile, res in selected_refit_results.items():
    row = selected_configs_df[
        selected_configs_df["selection_profile"].eq(profile)
    ].iloc[0]

    model_family = str(row["model_family"])
    object_threshold = float(row["object_threshold"])

    if model_family == "standard_rule":
        obj_df = res["object_tables"][object_threshold]
        validation_object_tables[profile] = obj_df

    elif model_family == "empirical_cv_rule":
        # empirical result contains a nested result keyed by base config
        nested_results = res["results"]
        if len(nested_results) == 0:
            print("[WARNING] No nested empirical result for", profile)
            continue

        first_key = next(iter(nested_results.keys()))
        rule_variant = str(row["rule_variant"])
        obj_df = nested_results[first_key]["object_tables_by_rule"][rule_variant]
        validation_object_tables[profile] = obj_df

print("Extracted validation object tables:")
for profile, obj_df in validation_object_tables.items():
    print(profile, obj_df.shape)
    display(obj_df.head())

In [ ]:
selected_validation_metrics_rows = []

for profile, obj_df in validation_object_tables.items():
    metrics = binary_detection_metrics(
        obj_df,
        true_col="true_peanut_object",
        pred_col="predicted_peanut_object",
    )

    row = {
        "selection_profile": profile,
    }
    row.update(metrics)

    selected_validation_metrics_rows.append(row)

selected_validation_metrics_df = pd.DataFrame(selected_validation_metrics_rows)
selected_validation_metrics_df = sort_simca_selection(selected_validation_metrics_df)

display(selected_validation_metrics_df)

In [ ]:
selected_metrics_by_batch = []
selected_metrics_by_image = []

for profile, obj_df in validation_object_tables.items():
    if "batch" in obj_df.columns:
        batch_df = metrics_by_group(
            obj_df,
            group_col="batch",
            true_col="true_peanut_object",
            pred_col="predicted_peanut_object",
        )
        batch_df["selection_profile"] = profile
        selected_metrics_by_batch.append(batch_df)

    if "source_image" in obj_df.columns:
        image_df = metrics_by_group(
            obj_df,
            group_col="source_image",
            true_col="true_peanut_object",
            pred_col="predicted_peanut_object",
        )
        image_df["selection_profile"] = profile
        selected_metrics_by_image.append(image_df)

selected_metrics_by_batch_df = (
    pd.concat(selected_metrics_by_batch, ignore_index=True)
    if selected_metrics_by_batch
    else pd.DataFrame()
)

selected_metrics_by_image_df = (
    pd.concat(selected_metrics_by_image, ignore_index=True)
    if selected_metrics_by_image
    else pd.DataFrame()
)

display(selected_metrics_by_batch_df)
display(selected_metrics_by_image_df)

In [ ]:
selected_metrics_by_batch = []
selected_metrics_by_image = []

for profile, obj_df in validation_object_tables.items():
    if "batch" in obj_df.columns:
        batch_df = metrics_by_group(
            obj_df,
            group_col="batch",
            true_col="true_peanut_object",
            pred_col="predicted_peanut_object",
        )
        batch_df["selection_profile"] = profile
        selected_metrics_by_batch.append(batch_df)

    if "source_image" in obj_df.columns:
        image_df = metrics_by_group(
            obj_df,
            group_col="source_image",
            true_col="true_peanut_object",
            pred_col="predicted_peanut_object",
        )
        image_df["selection_profile"] = profile
        selected_metrics_by_image.append(image_df)

selected_metrics_by_batch_df = (
    pd.concat(selected_metrics_by_batch, ignore_index=True)
    if selected_metrics_by_batch
    else pd.DataFrame()
)

selected_metrics_by_image_df = (
    pd.concat(selected_metrics_by_image, ignore_index=True)
    if selected_metrics_by_image
    else pd.DataFrame()
)

display(selected_metrics_by_batch_df)
display(selected_metrics_by_image_df)

In [ ]:
three_way_tables = {}

for profile, obj_df in validation_object_tables.items():
    obj_3way = add_three_way_object_decision(
        obj_df,
        target_class="peanut",
        lower_threshold=0.40,
        upper_threshold=float(
            selected_configs_df.loc[
                selected_configs_df["selection_profile"].eq(profile),
                "object_threshold",
            ].iloc[0]
        ),
    )

    three_way_tables[profile] = obj_3way

    print("=" * 80)
    print(profile)
    display(
        obj_3way
        .groupby(["decision_3way", "true_label_object"], dropna=False)
        .size()
        .reset_index(name="n_objects")
    )

In [ ]:
validation_error_rows = []

for profile, obj_df in validation_object_tables.items():
    err = obj_df.dropna(
        subset=["true_peanut_object", "predicted_peanut_object"]
    ).copy()

    err = err[
        err["true_peanut_object"].astype(bool)
        != err["predicted_peanut_object"].astype(bool)
    ].copy()

    if len(err) == 0:
        continue

    err["selection_profile"] = profile
    validation_error_rows.append(err)

validation_errors_df = (
    pd.concat(validation_error_rows, ignore_index=True, sort=False)
    if validation_error_rows
    else pd.DataFrame()
)

display(validation_errors_df)

VALIDATION_ERRORS_PATH= RESULTS_DIR / "selected_validation_object_errors.parquet"
save_parquet(validation_errors_df, VALIDATION_ERRORS_PATH)

print("Saved validation object errors:")
print(VALIDATION_ERRORS_PATH)

In [ ]:
# Pixel-level diagnostics are easier for standard-rule refits because they keep `pixel_df`.
# For empirical refits, pixel tables are stored inside the nested empirical result.

pixel_tables_by_profile = {}

for profile, res in selected_refit_results.items():
    row = selected_configs_df[
        selected_configs_df["selection_profile"].eq(profile)
    ].iloc[0]

    model_family = str(row["model_family"])

    if model_family == "standard_rule":
        pixel_tables_by_profile[profile] = res["pixel_df"]

    elif model_family == "empirical_cv_rule":
        nested_results = res["results"]
        if len(nested_results) == 0:
            continue

        first_key = next(iter(nested_results.keys()))
        stored = nested_results[first_key]

        if "pixel_variants_df" in stored:
            rule_variant = str(row["rule_variant"])
            pix = stored["pixel_variants_df"].copy()
            pix["predicted_peanut_pixel"] = pix[f"pred_{rule_variant}"].astype(bool)
            pix["rule_statistic"] = pix[f"stat_{rule_variant}"]
            pix["rule_limit"] = pix[f"limit_{rule_variant}"]
            pix["rule_name"] = rule_variant
            pixel_tables_by_profile[profile] = pix

print("Pixel tables available:")
for profile, pix in pixel_tables_by_profile.items():
    print(profile, pix.shape)

In [ ]:
pixel_error_summaries = []

for profile, pix in pixel_tables_by_profile.items():
    err_img = summarize_pixel_errors_by_image(pix)
    if len(err_img) > 0:
        err_img["selection_profile"] = profile
        pixel_error_summaries.append(err_img)

pixel_errors_by_image_df = (
    pd.concat(pixel_error_summaries, ignore_index=True)
    if pixel_error_summaries
    else pd.DataFrame()
)

display(pixel_errors_by_image_df)

PIXEL_ERRORS_BY_IMAGE_PATH = RESULTS_DIR / "selected_validation_pixel_errors_by_image.parquet"
save_parquet(pixel_errors_by_image_df, PIXEL_ERRORS_BY_IMAGE_PATH)

print("Saved pixel errors by image:")
print(PIXEL_ERRORS_BY_IMAGE_PATH)

In [ ]:
if len(pixel_tables_by_profile) > 0:
    best_profile = selected_configs_df.iloc[0]["selection_profile"]

    if best_profile not in pixel_tables_by_profile:
        best_profile = next(iter(pixel_tables_by_profile.keys()))

    best_pixel_df = pixel_tables_by_profile[best_profile]

    validation_image_keys = (
        best_pixel_df["source_image"]
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

    print("Best profile for pixel overlays:", best_profile)
    print("Validation images:", validation_image_keys)

    for image_key in validation_image_keys[:4]:
        print("Image:", image_key)

        try:
            plot_pixel_error_overlay(
                image_key=image_key,
                image_db=image_db,
                pixel_df=best_pixel_df,
                target_class="peanut",
                base="image_ref",
                title=f"Validation pixel errors — {best_profile} — {image_key}",
                show=True,
            )

            plot_pixel_fp_fn_overlay(
                image_key=image_key,
                image_db=image_db,
                pixel_df=best_pixel_df,
                target_class="peanut",
                base="image_ref",
                title=f"Validation FP/FN pixels — {best_profile} — {image_key}",
                show=True,
            )

        except Exception as exc:
            print(f"[WARNING] Could not plot pixel overlay for {image_key}: {exc!r}")
else:
    print("No pixel table available for overlays.")

In [ ]:
# save_parquet(selected_validation_metrics_df, SELECTED_VALIDATION_METRICS_PATH)
# save_parquet(selected_metrics_by_batch_df, SELECTED_VALIDATION_BATCH_METRICS_PATH)
# save_parquet(selected_metrics_by_image_df, SELECTED_VALIDATION_IMAGE_METRICS_PATH)
# save_parquet(validation_errors_df, VALIDATION_ERRORS_PATH)
# save_parquet(pixel_errors_by_image_df, PIXEL_ERRORS_BY_IMAGE_PATH)

In [ ]:
selection_config = {
    "db_h5_path": str(DB_H5_PATH),
    "results_dir": str(RESULTS_DIR),

    "target_class": TARGET_CLASS,
    "train_batches": TRAIN_BATCHES,
    "validation_batches": VALIDATION_BATCHES,
    "final_test_batches_reserved": FINAL_TEST_BATCHES,
    "use_mixtures_for_selection": USE_MIXTURES_FOR_SELECTION,

    "train_filters": train_filters,
    "validation_filters": validation_filters,
    "final_test_filters_reserved": final_test_filters,
    "mixture_filters_reserved": mixture_filters,

    "matrix_methods_standard": MATRIX_METHODS_STANDARD,
    "matrix_method_empirical": MATRIX_METHOD_EMPIRICAL,
    "balanced_pixel_strategies": BALANCED_PIXEL_STRATEGIES,
    "m_balanced_pixels": int(M_BALANCED_PIXELS),
    "replace_balanced_pixels": bool(REPLACE_BALANCED_PIXELS),
    "random_state": int(RANDOM_STATE),

    "n_components_values_standard": N_COMPONENTS_VALUES_STANDARD,
    "n_components_values_empirical": N_COMPONENTS_VALUES_EMPIRICAL,
    "alpha_values_standard": ALPHA_VALUES_STANDARD,
    "alpha_empirical": float(ALPHA_EMPIRICAL),
    "standard_rule_names": STANDARD_RULE_NAMES,
    "empirical_rule_variants": EMPIRICAL_RULE_VARIANTS,
    "object_thresholds": OBJECT_THRESHOLDS,

    "position_dilation_radius": int(POSITION_DILATION_RADIUS),
    "sg_window_length": int(SG_WINDOW_LENGTH),
    "sg_polyorder": int(SG_POLYORDER),
    "cv_n_splits": CV_N_SPLITS,
    "cv_group_col": CV_GROUP_COL,

    "preprocessing_configs": {
        name: list(steps)
        for name, steps in preprocessing_configs.items()
    },
}

with open(SELECTION_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(selection_config, f, indent=2)

print("Saved notebook config:")
print(SELECTION_CONFIG_JSON)